<a href="https://colab.research.google.com/github/dsdsgege/vision-robustness-analyzing/blob/dani_dev/Projekt_munka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Adathalmazok kialakítása + augmentáció  

In [1]:
import torch
import numpy as np
from torchvision import datasets
from torchvision.transforms import v2 as transforms
from torch.utils.data import DataLoader, random_split, Dataset

# Segédosztály a transzformációk különválasztásához Subset esetén
class ApplyTransform(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)


# Külön osztály a zaj-augmentációhoz
class NoiseAugmentation:
    def __init__(self, noise_type=None, **noise_params):
        """
        noise_type: 'gaussian', 'salt_pepper', 'poisson', vagy None
        noise_params: a zajtípushoz tartozó paraméterek (pl. mean, std, amount)
        """
        self.noise_type = noise_type
        self.noise_params = noise_params

    def __call__(self, tensor):
        if self.noise_type is None:
            return tensor

        if self.noise_type == 'gaussian':
            mean = self.noise_params.get('mean', 0.0)
            std = self.noise_params.get('std', 0.1)
            noise = torch.randn_like(tensor) * std + mean
            return torch.clamp(tensor + noise, 0.0, 1.0)

        elif self.noise_type == 'salt_pepper':
            amount = self.noise_params.get('amount', 0.05)
            s_vs_p = self.noise_params.get('salt_vs_pepper', 0.5)
            noisy = tensor.clone()
            # Véletlenszerű mátrix a valószínűségekhez
            rand_matrix = torch.rand_like(tensor)
            # Só (fehér pixelek)
            noisy[rand_matrix < amount * s_vs_p] = 1.0
            # Bors (fekete pixelek)
            noisy[rand_matrix > 1.0 - amount * (1.0 - s_vs_p)] = 0.0
            return noisy

        elif self.noise_type == 'poisson':
            # A Poisson eloszlás gyakoriság alapú, így a 0-1 tartományt felskálázzuk
            # egy "rate" paraméterrel (pl. 255), rátesszük a zajt, majd visszaskálázzuk.
            scale = self.noise_params.get('scale', 255.0)
            noisy = torch.poisson(tensor * scale) / scale
            return torch.clamp(noisy, 0.0, 1.0)

        else:
            raise ValueError(f"Nem támogatott zaj típus: {self.noise_type}")


class DataManager:
    def __init__(self, dataset_name='cifar10', data_dir='./data', batch_size=100,
                 valid_split=0.2, num_workers=2, noise_type=None, noise_params=None):
        self.dataset_name = dataset_name.lower()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.valid_split = valid_split
        self.num_workers = num_workers

        # Zaj paraméterek elmentése
        self.noise_type = noise_type
        self.noise_params = noise_params if noise_params is not None else {}

        self.datasets_dict = {
            'mnist': datasets.MNIST,
            'cifar10': datasets.CIFAR10,
            'cifar100': datasets.CIFAR100,
        }

        if self.dataset_name not in self.datasets_dict:
            raise ValueError(f"Nem támogatott adathalmaz: {self.dataset_name}")

        self.DatasetClass = self.datasets_dict[self.dataset_name]

    def _get_mean_std(self):
        """
        Adathalmaz-specifikus normalizációs értékek.
        Fontos: Az MNIST 1 csatornás, a többi 3 csatornás!
        """
        if self.dataset_name == 'mnist':
            return [0.1307], [0.3081]
        elif self.dataset_name == 'cifar10':
            return [0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010]
        elif self.dataset_name == 'cifar100':
            return [0.5071, 0.4865, 0.4409], [0.2673, 0.2564, 0.2762]
        return [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]

    def get_loaders(self):
        mean, std = self._get_mean_std()

        # A tréning során adjuk hozzá a zajt a 0-1 skálázás után, normalizáció előtt
        train_transform = transforms.Compose([
            transforms.ToImage(),
            transforms.ToDtype(torch.float32, scale=True),
            NoiseAugmentation(noise_type=self.noise_type, **self.noise_params),
            transforms.Normalize(mean=mean, std=std)
        ])

        # A validációnál és tesztnél NINCS zaj augmentáció
        test_transform = transforms.Compose([
            transforms.ToImage(),
            transforms.ToDtype(torch.float32, scale=True),
            transforms.Normalize(mean=mean, std=std)
        ])

        # Az alap adathalmazt transzformáció nélkül töltjük be a train/valid szétválasztáshoz
        full_train_set = self.DatasetClass(root=self.data_dir, train=True, download=True)
        test_set = self.DatasetClass(root=self.data_dir, train=False, download=True, transform=test_transform)

        valid_size = int(len(full_train_set) * self.valid_split)
        train_size = len(full_train_set) - valid_size

        # Logikailag helyes split fix seed-del
        train_subset, valid_subset = random_split(
            full_train_set, [train_size, valid_size],
            generator=torch.Generator().manual_seed(44)
        )

        # Itt alkalmazzuk külön a transzformációkat
        train_data = ApplyTransform(train_subset, transform=train_transform)
        valid_data = ApplyTransform(valid_subset, transform=test_transform)

        train_loader = DataLoader(train_data, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
        valid_loader = DataLoader(valid_data, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)
        test_loader = DataLoader(test_set, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

        return train_loader, valid_loader, test_loader


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

#Adatok beolvasása, MNIST, CIFAR10, CIFAR100 és ImageNet közül lehet választani.
#-----------------------------------------------------------------------------------------------------------
# Gaussian zajjal
#data_manager = DataManager(dataset_name='cifar10', noise_type='gaussian', noise_params={'std': 0.06}) # std nagyobb --> nagyobb zaj
#-----------------------------------------------------------------------------------------------------------
# Só-bors zajjal
#data_manager = DataManager(dataset_name='cifar10', noise_type='salt_pepper', noise_params={'amount': 0.06}) # amount --> aránya az eltorított pixeleknek
#-----------------------------------------------------------------------------------------------------------
# Poisson zajjal
data_manager = DataManager(dataset_name='cifar10', noise_type='poisson', noise_params={'scale': 50.0}) # normalizált érték felskálázása --> kisebb érték = nagyobb zaj
#-----------------------------------------------------------------------------------------------------------

train_loader, valid_loader, test_loader = data_manager.get_loaders()

print(f"Tanító halmaz mérete: {len(train_loader.dataset)}")
print(f"Validációs halmaz mérete: {len(valid_loader.dataset)}")
print(f"Teszt halmaz mérete: {len(test_loader.dataset)}")

print(f"Batchenkénti képek száma: {len(train_loader)}")

# Egy batch lekérése teszteléshez
train_images, train_labels = next(iter(train_loader))
print(f"Képek dimenziója: {train_images.shape}, Címkék: {train_labels.shape}")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display
from torchvision.transforms import v2 as transforms

def imshow_single(img, title, mean, std):
    """Segédfüggvény a visszanormalizáláshoz és a tengelyre rajzoláshoz."""
    mean_tensor = torch.tensor(mean).view(len(mean), 1, 1)
    std_tensor = torch.tensor(std).view(len(std), 1, 1)

    img = img.clone() * std_tensor + mean_tensor
    img = torch.clamp(img, 0, 1)

    # 1 csatornás (MNIST) és 3 csatornás (CIFAR/ImageNet) kezelése
    npimg = img.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap='gray')
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))

    plt.title(title)
    plt.axis('off')

# 1. Paraméterek és nyers adathalmaz kinyerése a létező data_manager-ből
mean, std = data_manager._get_mean_std()
raw_dataset = data_manager.DatasetClass(root=data_manager.data_dir, train=False, download=True)

# Zaj paraméterek kiolvasása (kezeli a noise_kwargs / noise_params elnevezéseket is)
current_noise_type = data_manager.noise_type
current_noise_params = getattr(data_manager, 'noise_kwargs', getattr(data_manager, 'noise_params', {}))

# 2. Tiszta transzformáció
clean_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(mean=mean, std=std)
])

# 3. Zajos transzformáció az aktuális DataManager beállításokkal
noisy_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    NoiseAugmentation(noise_type=current_noise_type, **current_noise_params),
    transforms.Normalize(mean=mean, std=std)
])

# 4. Megjelenítő függvény, amit a widget hív
def show_current_noise(image_index=0):
    raw_image, label = raw_dataset[image_index]

    clean_tensor = clean_transform(raw_image)
    noisy_tensor = noisy_transform(raw_image)

    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    imshow_single(clean_tensor, title=f"Tiszta kép (Index: {image_index})", mean=mean, std=std)

    plt.subplot(1, 2, 2)
    zaj_nev = current_noise_type.upper() if current_noise_type else "NINCS ZAJ"
    imshow_single(noisy_tensor, title=f"Alkalmazott zaj: {zaj_nev}", mean=mean, std=std)

    plt.tight_layout()
    plt.show()

# 5. Widget beállítása csak a kép léptetésére (sima beviteli mező)
print(f"Betöltött zaj konfiguráció: {current_noise_type} {current_noise_params}")

widgets.interact(
    show_current_noise,
    image_index=widgets.BoundedIntText(
        min=0,
        max=len(raw_dataset)-1,
        step=1,
        value=0,
        description='Kép Index:'
    )
)

#Reziduális konvolúciós neurális háló


In [2]:
import torch.nn as nn
import torch.nn.functional as F

class ResNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResNetBlock, self).__init__()
        # Conv Layer 1
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)

        # Conv Layer 2
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Ha a felbontás csökken (stride!=1) VAGY a csatornaszám nő, a maradékot (residual) is igazítani kell
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = x  # Eredeti bemenet elmentése

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)

        if self.downsample:
            residual = self.downsample(residual)

        x += residual # Skip connection
        x = F.relu(x)

        return x

def make_basic_block_layer(in_channels, out_channels, num_blocks, stride=1):
    layers = []
    layers.append(ResNetBlock(in_channels, out_channels, stride))
    for _ in range(1, num_blocks):
        layers.append(ResNetBlock(out_channels, out_channels))
    return nn.Sequential(*layers)

In [3]:
class Residual_CNN(nn.Module):
    def __init__(self, num_blocks, filters=[64, 128, 256], num_classes=100, drop_rate=0.3, input_channels=3):
        """
        num_blocks: Lista, ami megmondja, hány blokk legyen egy-egy rétegcsoportban (pl. [3, 3, 3])
        channels: Lista, ami a csatornák (vastagság) számát adja meg az egyes csoportokban
        """
        super(Residual_CNN, self).__init__()

        if len(num_blocks) != len(filters):
            raise ValueError("A num_blocks és a channels listák hossza meg kell egyezzen!")

        # A legelső konvolúció kimeneti csatornaszáma (a lista első eleme)
        self.in_channels = filters[0]

        # Kezdeti konvolúciós réteg (kezeli az eltérő bementeti csatornaszámot --> fekete-fehér = 1| RGB = 3)
        self.conv1 = nn.Conv2d(input_channels, self.in_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(self.in_channels)
        self.dropout1 = nn.Dropout(drop_rate)

        # Dinamikus rétegek létrehozása (nn.ModuleList segítségével)
        self.layers = nn.ModuleList()
        for i in range(len(filters)):
            out_channels = filters[i]
            # Az első csoportnál (i==0) nincs kicsinyítés, a többinél viszont felezzük a felbontást
            stride = 1 if i == 0 else 2

            self.layers.append(
                make_basic_block_layer(self.in_channels, out_channels, num_blocks[i], stride=stride)
            )
            # Frissítjük a bemeneti csatornaszámot a következő ciklushoz
            self.in_channels = out_channels

        # Global Average Pooling és FC réteg
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(drop_rate)

        # A lineáris réteg bemenete mindig a legutolsó megadott csatornaszám lesz
        self.fc = nn.Linear(filters[-1], num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout1(x)

        # Végigmegyünk az összes dinamikusan legenerált rétegcsoporton
        for layer in self.layers:
            x = layer(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)

        return x

#Tanítás

In [4]:
from tqdm.auto import tqdm

# ---------------- EARLY STOPPING OSZTÁLY ----------------
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        if self.best_model_state is not None:
            model.load_state_dict(self.best_model_state)

# ------------------------------- TRAIN & TEST FÜGGVÉNYEK -------------------------------
def train(dataloader, model, loss_fn, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = loss_fn(outputs, y)
        loss.backward()
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

    avg_loss = running_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def test(dataloader, model, loss_fn, device, mode="Test"):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = loss_fn(outputs, y)
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    avg_loss = running_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# -------------------------------------------------------------- TRAIN_AND_EVAL FÜGGVÉNY --------------------------------------------------------------
def train_and_eval(model, train_loader, valid_loader, optimizer, device, epochs=30, lr=1e-3, patience=5, use_scheduler=False):
    loss_fn = nn.CrossEntropyLoss()
    early_stopping = EarlyStopping(patience=patience, delta=0.001) # delta: Mennyit kell minimum javulnia

    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs
        )
    else:
        scheduler = None

    hist = {'acc': [], 'val_acc': [], 'loss': [], 'val_loss': []}
    pbar = tqdm(range(epochs), position=1, leave=False, desc="Aktuális tanítás")

    for epoch in pbar:
        # Tanítás és Validáció
        loss, acc = train(train_loader, model, loss_fn, optimizer, scheduler, device)
        val_loss, val_acc = test(valid_loader, model, loss_fn, device, "Validation")

        # Elmentjük a görbéket
        hist["loss"].append(loss)
        hist["acc"].append(acc)
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(val_acc)

        # Kiírjuk az aktuális állapotot a folyamatsávra
        pbar.set_postfix(train_acc=f"{acc:.1f}%", train_loss=f"{loss:.3f}", val_acc=f"{val_acc:.1f}%", val_loss=f"{val_loss:.3f}")

        # ----------- EARLY STOPPING ELLENŐRZÉS -----------
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            tqdm.write(f"     => Korai leállítás aktiválva a(z) {epoch+1}. epochnál (Val Loss: {val_loss:.4f})")
            break # Kilépünk az epoch ciklusból

    # Legjobb súlyok visszatöltése!
    early_stopping.load_best_model(model)

    return hist

In [5]:
def get_fresh_model(dataset_name, device, lr=1e-3):
    """Minden kísérlethez egy teljesen új, tiszta modellt kell indítanunk!"""

    if dataset_name == 'mnist':
        model = Residual_CNN( # ResNet 8
            num_blocks=[1, 1, 1],
            filters=[8, 16, 32],
            num_classes=10,
            input_channels=1,
            drop_rate=0.1
        ).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    elif dataset_name == 'cifar10':
        model = Residual_CNN( # ResNet 20
            num_blocks=[3, 3, 3],
            filters=[16, 32, 64],
            num_classes=10,
            input_channels=3,
            drop_rate=0.2
        ).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    elif dataset_name == 'cifar100':
        model = Residual_CNN(  # ResNet 34
            num_blocks=[3, 4, 6, 3],
            filters=[64, 128, 256, 512],
            num_classes=100,
            input_channels=3,
            drop_rate=0.4
        ).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)

    else:
        raise ValueError(f"Ismeretlen adathalmaz: {dataset_name}")

    return model, optimizer

In [ ]:
import os
import pickle
import random
import numpy as np
import torch
from google.colab import drive
from tqdm.auto import tqdm

# =====================================================================
# GOOGLE DRIVE CSATLAKOZTATÁSA
# =====================================================================
drive.mount('/content/drive')

# Mentési útvonal megadása (Nevezd át nyugodtan a mappát/fájlt)
save_path = '/content/drive/MyDrive/Projekt_munka_histories.pkl'

# =====================================================================
# GLOBÁLIS SEED BEÁLLÍTÁSA A TELJES REPRODUKÁLHATÓSÁGÉRT
# =====================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # Multi-GPU esetén
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Meghívjuk az inicializálás előtt
set_seed(42)

# =====================================================================
# KORÁBBI EREDMÉNYEK BETÖLTÉSE (ÚJRAINDÍTHATÓSÁG)
# =====================================================================
if os.path.exists(save_path):
    print(f"Korábbi mentés megtalálva! Eredmények betöltése: {save_path}")
    with open(save_path, 'rb') as f:
        all_histories = pickle.load(f)
else:
    print("Nem található korábbi mentés. Tiszta lappal indulunk.")
    all_histories = {}


# --- KONFIGURÁCIÓK ---
noise_configs = {
    'gaussian':    {'std':    [0.1, 0.2, 0.3, 0.4, 0.5]},
    'salt_pepper': {'amount': [0.2, 0.3, 0.4, 0.5, 0.6]},
    'poisson':     {'scale':  [800.0, 650.0, 500.0, 350.0, 200.0]}
}

dataset_names = ['mnist', 'cifar10', 'cifar100']

epochs = 30
lr = 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"

total_experiments = len(dataset_names) * sum(len(list(d.values())[0]) for d in noise_configs.values())
global_pbar = tqdm(total=total_experiments, desc="ÖSSZESÍTETT HALADÁS", position=0, leave=True, colour='green')

# =====================================================================
# A FŐ CIKLUSOK
# =====================================================================
for dataset in dataset_names:
    tqdm.write(f"\n{'='*50}")
    tqdm.write(f"AKTUÁLIS ADATBÁZIS: {dataset.upper()}")
    tqdm.write(f"{'='*50}")

    # Biztosítjuk, hogy a kulcs létezzen
    if dataset not in all_histories:
        all_histories[dataset] = {}

    for noise_type, param_dict in noise_configs.items():
        tqdm.write(f"\n---> Zaj típus: {noise_type.upper()}")

        if noise_type not in all_histories[dataset]:
            all_histories[dataset][noise_type] = {}

        param_key = list(param_dict.keys())[0]
        values_list = param_dict[param_key]

        for val in values_list:

            # ---------------- ÚJRAINDÍTHATÓSÁGI LOGIKA ----------------
            # Ha ez a konkrét beállítás már megvan a szótárban, átugorjuk!
            if val in all_histories[dataset][noise_type]:
                tqdm.write(f"     [UGRÁS] Ez a szint már készen van: {dataset} | {noise_type} | {val}")
                global_pbar.update(1)
                continue
            # -----------------------------------------------------

            params = {param_key: val}
            global_pbar.set_postfix(dataset=dataset, noise=noise_type, level=val)
            tqdm.write(f"     Szint indítása: {params}")

            # Adatok betöltése
            data_manager = DataManager(
                dataset_name=dataset,
                batch_size=128,
                noise_type=noise_type,
                noise_params=params
            )
            train_loader, valid_loader, test_loader = data_manager.get_loaders()

            # FRISS modell és optimizer
            model, optimizer = get_fresh_model(dataset, device, lr=lr)

            # Tanítás
            history = train_and_eval(
                model=model,
                train_loader=train_loader,
                valid_loader=valid_loader,
                optimizer=optimizer,
                device=device,
                epochs=epochs,
                lr=lr,
                patience=5,
                use_scheduler=False
            )

            # Eredmény elmentése a memóriába
            all_histories[dataset][noise_type][val] = history

            # ---------------- MENTÉS DRIVE-RA ----------------
            # Minden sikeres lépés után fizikailag is kiírjuk a fájlt
            with open(save_path, 'wb') as f:
                pickle.dump(all_histories, f)
            tqdm.write(f"     [MENTVE] Eredmény biztonságosan kiírva a Drive-ra.")
            # -------------------------------------------------

            global_pbar.update(1)

global_pbar.close()
tqdm.write("\n🎉 Az összes kísérlet sikeresen lefutott!")

Mounted at /content/drive
Nem található korábbi mentés. Tiszta lappal indulunk.


ÖSSZESÍTETT HALADÁS:   0%|          | 0/45 [00:00<?, ?it/s]


AKTUÁLIS ADATBÁZIS: MNIST

---> Zaj típus: GAUSSIAN
     Szint indítása: {'std': 0.1}



  0%|          | 0.00/9.91M [00:00<?, ?B/s]
  0%|          | 32.8k/9.91M [00:00<01:04, 153kB/s]
  1%|          | 98.3k/9.91M [00:00<00:40, 240kB/s]
  2%|▏         | 164k/9.91M [00:00<00:36, 267kB/s] 
  4%|▎         | 360k/9.91M [00:00<00:18, 517kB/s]
  6%|▋         | 623k/9.91M [00:01<00:12, 763kB/s]
 13%|█▎        | 1.28M/9.91M [00:01<00:05, 1.52MB/s]
 23%|██▎       | 2.33M/9.91M [00:01<00:02, 2.59MB/s]
 47%|████▋     | 4.65M/9.91M [00:01<00:01, 5.15MB/s]
100%|██████████| 9.91M/9.91M [00:01<00:00, 5.02MB/s]

  0%|          | 0.00/28.9k [00:00<?, ?B/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]

  0%|          | 0.00/1.65M [00:00<?, ?B/s]
  2%|▏         | 32.8k/1.65M [00:00<00:10, 154kB/s]
  6%|▌         | 98.3k/1.65M [00:00<00:06, 242kB/s]
 10%|▉         | 164k/1.65M [00:00<00:05, 270kB/s] 
 24%|██▍       | 393k/1.65M [00:00<00:02, 582kB/s]
 40%|███▉      | 655k/1.65M [00:01<00:01, 804kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]

100%|██████████| 4.54k/4.54k [0

Aktuális tanítás:   0%|          | 0/30 [00:00<?, ?it/s]